# **Olist E-Commerce Data Cleaning & Preparation**

> **Summary:** This notebook cleans and prepares the nine Olist datasets for analysis while preserving the original raw data. It standardizes timestamps, translates product categories, handles duplicate reviews, aggregates geolocation data, and creates order-level and item-level analytical features. All orders are retained, with flags defining eligibility for revenue analysis and the analysis period, while invalid timestamp pairs are excluded only from the affected delivery metrics. Eight processed CSV files are saved and validated in `data/processed/`.

---

## **Purpose of This Notebook**

In Stage 1, I explored the nine Olist datasets and identified their structure, relationships, missing values, and potential data-quality issues.

The goal of this notebook is to clean and prepare the datasets for further analysis while preserving the original raw data. I handle missing values carefully, standardize data types, investigate unusual records, and create useful features for business analysis.

The processed datasets will be used in the next stages for PostgreSQL, SQL-based business analysis, and Power BI dashboard development.

> **First understand the data. Then clean it. Then analyze it.**

---

## **What I Want to Achieve**

During this stage, I aim to:

- Convert timestamp columns to appropriate datetime formats.
- Handle missing product categories and translate category names into English.
- Investigate order statuses, duplicate reviews, and timestamp inconsistencies.
- Aggregate geolocation data to one record per ZIP code prefix before joining.
- Create analytical features for delivery performance, revenue, customer satisfaction, and seller-customer geographic comparisons.
- Preserve relevant records while defining appropriate rules for revenue and delivery analysis.
- Validate the processed datasets, check row counts, and reconcile item-price totals against the original data.

---

## **My Approach**

For each important cleaning step, I want to understand:

**What needs cleaning?**

**Why does it matter?**

**What action should I take?**

**How can I verify the result?**

I document important decisions, preserve the original raw data, and avoid deleting or replacing values without a clear reason.

---

## **Expected Outcome**

By the end of this notebook, I will have cleaned, documented, and validated datasets ready for database loading, SQL analysis, and further business analysis.

> **Important principle:** Every cleaning decision should improve data reliability without introducing unnecessary data loss.

## **Step 1: Set Up the Data Cleaning Environment**

In this step, I will import the required libraries, connect Google Drive, and configure the project directories for data cleaning.

The original datasets stored in `data/raw/` will remain unchanged. All cleaning operations will be performed on working copies, and the validated datasets will be saved in `data/processed/`.

The code will also verify that the raw data directory exists before proceeding, helping prevent file path errors during data loading.

### **Connecting Google Drive and Configuring Project Paths**

Google Drive is connected to Google Colab to access the Olist dataset files stored in my Drive. The code checks whether Drive is already mounted and mounts it only if necessary.

The project and raw data directories are then configured, and the existence of the raw data directory is verified before continuing.

In [69]:

import os
import pandas as pd
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

project_path = (
    "/content/drive/MyDrive/Data Science/"
    "Olist E-Commerce Analytics Project"
)

raw_path = os.path.join(project_path, "data", "raw")
processed_path = os.path.join(project_path, "data", "processed")

if not os.path.isdir(raw_path):
    raise FileNotFoundError(f"Raw data directory not found: {raw_path}")

os.makedirs(processed_path, exist_ok=True)

print("Data cleaning environment set up successfully.")
print(f"Raw data directory: {raw_path}")
print(f"Processed data directory: {processed_path}")

Data cleaning environment set up successfully.
Raw data directory: /content/drive/MyDrive/Data Science/Olist E-Commerce Analytics Project/data/raw
Processed data directory: /content/drive/MyDrive/Data Science/Olist E-Commerce Analytics Project/data/processed


### **Observation**

The required libraries were imported, and Google Drive was connected successfully. The project directories were configured, and the raw data directory was verified.

The `data/processed/` directory was created to store cleaned datasets, while the original CSV files in `data/raw/` remain unchanged.

## **Step 2: Load the Raw Datasets**

The Olist project contains nine related datasets covering orders, customers, order items, payments, reviews, products, sellers, geolocation, and product category translations.

In this step, I will load all nine CSV files into a dictionary of Pandas DataFrames. The code will verify that each expected file exists before loading it and summarize the dimensions of each dataset to confirm successful loading.

The original CSV files in `data/raw/` will remain unchanged throughout the cleaning process.

In [70]:
# Define the expected raw dataset files
files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

# Load each CSV into the data dictionary
data = {}

for name, filename in files.items():
    file_path = os.path.join(raw_path, filename)

    # Stop if an expected dataset is missing
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"Dataset not found: {file_path}")

    # Load the raw CSV without modifying the original file
    data[name] = pd.read_csv(file_path)

# Confirm that all expected datasets have been loaded
assert len(data) == len(files), "Some datasets were not loaded."

print(f"Successfully loaded all {len(data)} datasets.")

Successfully loaded all 9 datasets.


In [71]:
# Summarize the dimensions of all loaded datasets
dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in data.items()
])

# Display the summary for a quick loading check
display(dataset_summary)

,Dataset,Rows,Columns
0,orders,99441,8
1,order_items,112650,7
2,payments,103886,5
3,reviews,99224,7
4,products,32951,9
5,sellers,3095,4
6,customers,99441,5
7,geolocation,1000163,5
8,translation,71,2


### **Observation**

All nine expected CSV datasets were loaded successfully into the `data` dictionary. The dimensions of each dataset were summarized to verify that the files were loaded correctly.

The original CSV files in `data/raw/` remain unchanged, and the loaded datasets are ready for the subsequent cleaning steps.

## **Step 3: Convert Datetime Columns**

In this step, I will convert the relevant timestamp columns in the orders, order items, and reviews datasets into Pandas datetime format. This will enable accurate date calculations for delivery performance, order timelines, and review handling in the subsequent cleaning steps. Invalid date values will be converted to `NaT` so they can be identified and handled during validation.

In [72]:
# Specify the timestamp columns that need conversion in each dataset.
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

In [73]:
# Convert the selected columns to datetime format.
# Invalid date values become NaT instead of causing errors.
for dataset_name, columns in datetime_columns.items():
    for column in columns:
        data[dataset_name][column] = pd.to_datetime(
            data[dataset_name][column],
            errors="coerce"
        )

print("Timestamp conversion completed successfully.")

Timestamp conversion completed successfully.


In [74]:
# Verify that each selected column has the correct datetime dtype
# and report missing values after conversion.
for dataset_name, columns in datetime_columns.items():
    print(f"\nDataset: {dataset_name}")

    for column in columns:
        dtype = data[dataset_name][column].dtype
        missing_count = data[dataset_name][column].isna().sum()

        print(
            f"{column}: dtype = {dtype}, "
            f"missing values = {missing_count}"
        )


Dataset: orders
order_purchase_timestamp: dtype = datetime64[ns], missing values = 0
order_approved_at: dtype = datetime64[ns], missing values = 160
order_delivered_carrier_date: dtype = datetime64[ns], missing values = 1783
order_delivered_customer_date: dtype = datetime64[ns], missing values = 2965
order_estimated_delivery_date: dtype = datetime64[ns], missing values = 0

Dataset: order_items
shipping_limit_date: dtype = datetime64[ns], missing values = 0

Dataset: reviews
review_creation_date: dtype = datetime64[ns], missing values = 0
review_answer_timestamp: dtype = datetime64[ns], missing values = 0


### **Observation**

The relevant timestamp columns in the orders, order items, and reviews datasets were successfully converted to Pandas datetime format. The data types and missing-value counts were verified to ensure the conversion was completed correctly.

Missing timestamps in the orders dataset were preserved for appropriate handling during subsequent cleaning and validation steps. No missing values were detected in the selected timestamp columns of the order items or reviews datasets. The original raw CSV files remain unchanged.

## **Step 4: Handle Missing Product Categories**

In this step, I will translate Portuguese product category names into English using the provided translation dataset. I will preserve the original category column and store the translated values in a new column named `product_category_english`.

Two categories that are missing from the supplied translation mapping will be translated manually. Only genuinely missing category values will be labeled as `unknown`, ensuring that valid product categories are not incorrectly classified as missing.

In [75]:
# Create a mapping from Portuguese category names to English names.
translation_map = (
    data["translation"]
    .set_index("product_category_name")["product_category_name_english"]
    .to_dict()
)

# Manually translate the two valid categories missing from the mapping.
translation_map.update({
    "pc_gamer": "gaming_pc",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "small_kitchen_appliances"
})

# Translate categories into a NEW column, preserving the original Portuguese names.
data["products"]["product_category_english"] = (
    data["products"]["product_category_name"].map(translation_map)
)

# Identify any non-missing categories that still lack a translation.
untranslated = data["products"].loc[
    data["products"]["product_category_name"].notna()
    & data["products"]["product_category_english"].isna(),
    "product_category_name"
].unique()

# Stop if any valid category remains untranslated.
if len(untranslated) > 0:
    raise ValueError(f"Untranslated categories found: {untranslated}")

# Label only genuinely missing category values as "unknown".
data["products"]["product_category_english"] = (
    data["products"]["product_category_english"].fillna("unknown")
)

print("Product category translation completed.")
print(
    "Products labeled unknown:",
    (data["products"]["product_category_english"] == "unknown").sum()
)

Product category translation completed.
Products labeled unknown: 610


### **Observation**

Product category names were successfully translated from Portuguese to English using the supplied translation dataset, with manual translations added for the two categories missing from the original mapping.

The original Portuguese category column was preserved, and a new `product_category_english` column was created. Only genuinely missing category values were labeled as `unknown`, resulting in **610 products with unknown categories** and no missing values in the translated category column.

All changes were applied to the working `products` DataFrame, while the original raw CSV files remained unchanged.

## **Step 5: Handle Missing Product Details**

The initial data exploration identified missing physical measurements and four products with zero recorded weight.

In this step, I will preserve missing product measurements as `NaN` and replace impossible zero product weights with `NaN`. The affected products will be retained so they remain available for analyses that do not require these measurements.

The original raw datasets will remain unchanged.

In [76]:
# Define the physical measurement columns used in this cleaning step.
measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

# Replace impossible zero weights with NaN rather than treating them as valid.
zero_weight_mask = data["products"]["product_weight_g"].eq(0)
zero_weight_count = int(zero_weight_mask.sum())

data["products"].loc[zero_weight_mask, "product_weight_g"] = float("nan")

# Preserve missing measurements as NaN; do not delete affected products.
print(f"Zero product weights replaced with NaN: {zero_weight_count}")
print(
    "Products with missing physical measurements:",
    data["products"][measurement_columns].isna().any(axis=1).sum()
)

# Verify that no zero weights remain.
assert not data["products"]["product_weight_g"].eq(0).any(), (
    "Some zero product weights remain."
)

print("Product measurement cleaning completed.")

Zero product weights replaced with NaN: 4
Products with missing physical measurements: 6
Product measurement cleaning completed.


### **Observations**

- 4 products had a recorded weight of `0` grams, which is an implausible physical measurement.
- All 4 zero product weights were replaced with `NaN` because reliable replacement values were unavailable.
- 6 products have at least one missing physical measurement across weight, length, height, or width.
- Missing measurements were preserved as null values, and no product records were deleted or assigned arbitrary replacement values.

These data quality issues will be considered in subsequent analyses involving product dimensions, weight, and freight costs.

## **Step 6: Handle Duplicate Reviews**

The review dataset may contain multiple reviews for the same order. In this step, I will retain the most recent review for each order by sorting the records using `review_creation_date` and `review_answer_timestamp`.

I will also flag reviews created before the corresponding purchase date rather than deleting them, since the project uses review scores rather than review timing. Finally, I will prepare an `is_low_review` indicator for low scores and exclude review comment text from the processed dataset because it is not required for the planned analysis.

In [77]:
# Create a working copy so the original reviews remain unchanged.
reviews_working = data["reviews"].copy()

# Add the purchase timestamp to identify reviews created before purchase.
# validate="many_to_one" ensures each order matches at most one order record.
reviews_working = reviews_working.merge(
    data["orders"][["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Flag suspicious review timing instead of deleting these records.
reviews_working["review_created_before_purchase"] = (
    reviews_working["review_creation_date"]
    < reviews_working["order_purchase_timestamp"]
)

# Sort newest reviews first; use the answer timestamp to break ties.
reviews_working = reviews_working.sort_values(
    ["review_creation_date", "review_answer_timestamp"],
    ascending=[False, False],
    na_position="last"
)

# Keep only the most recent review for each order.
# This produces one review record per order_id.
reviews_latest = reviews_working.drop_duplicates(
    subset="order_id",
    keep="first"
).copy()

# Mark scores of 1 or 2 as low reviews; other valid scores become 0.
# Missing scores remain missing rather than being treated as positive reviews.
reviews_latest["is_low_review"] = reviews_latest["review_score"].map(
    lambda score: 1 if score in [1, 2]
    else (0 if pd.notna(score) else float("nan"))
)

# Remove comment text because the planned analysis uses review scores,
# not free-text comments.
reviews_latest = reviews_latest.drop(
    columns=["review_comment_title", "review_comment_message",
             "order_purchase_timestamp"],
    errors="ignore"
)

print("Review cleaning and deduplication completed.")

Review cleaning and deduplication completed.


In [78]:
# Compare the original and cleaned review counts.
print("Original review rows:", len(data["reviews"]))
print("Cleaned review rows:", len(reviews_latest))
print(
    "Rows removed during deduplication:",
    len(data["reviews"]) - len(reviews_latest)
)

# Confirm that each order has at most one review in the cleaned dataset.
assert reviews_latest["order_id"].is_unique, (
    "Duplicate order IDs remain in the cleaned reviews."
)

# Check the number of reviews flagged as created before purchase.
early_review_count = reviews_latest[
    "review_created_before_purchase"
].sum()

print("Early reviews retained:", int(early_review_count))

# Check the distribution of low-review indicators.
print("\nLow-review indicator counts:")
print(reviews_latest["is_low_review"].value_counts(dropna=False))

# Confirm that unnecessary comment columns were removed.
comment_columns_remaining = [
    column for column in
    ["review_comment_title", "review_comment_message"]
    if column in reviews_latest.columns
]

assert not comment_columns_remaining, (
    f"Comment columns still present: {comment_columns_remaining}"
)

print("\nAll review validation checks passed.")

Original review rows: 99224
Cleaned review rows: 98673
Rows removed during deduplication: 551
Early reviews retained: 71

Low-review indicator counts:
is_low_review
0    84178
1    14495
Name: count, dtype: int64

All review validation checks passed.


### **Observations**

- The reviews dataset initially contained 99,224 records.
- No exact duplicate rows were identified, but multiple review records were associated with some orders.
- No missing values were found in the review creation or answer timestamps.
- The most recent review per order was retained based on review creation date, using the answer timestamp to resolve ties.
- The resulting `reviews_latest` dataset contains **98,673 records**, with 551 rows removed during deduplication.
- Among the retained reviews, 71 were flagged as having been created before the corresponding purchase. These records were retained rather than deleted.
- The `is_low_review` indicator was created to identify reviews with scores of 1 or 2, while missing review scores remain missing.
- Review comment title and message columns were excluded from the cleaned dataset because they are not required for the planned analysis.
- The original reviews dataset was preserved, and the cleaned records were stored separately in `reviews_latest`.

## **Step 7: Investigate Order Statuses and Delivery Timestamps**

Before creating delivery-related metrics, we need to understand the distribution of order statuses and assess the reliability of the available timestamps.

This step will:
- Examine the distribution of order statuses.
- Investigate missing delivery timestamps by order status.
- Identify timestamps that occur before the purchase timestamp.
- Establish rules for calculating delivery metrics without deleting orders or imputing arbitrary dates.

All original orders will be retained. Missing or invalid timestamps will be handled only when calculating metrics that depend on those timestamps.

In [79]:
# Count orders in each status.
# Keep every order; this is an investigation, not a filtering step.

status_counts = (
    data["orders"]["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

display(status_counts)

print("Total orders:", len(data["orders"]))
print("Unique order IDs:", data["orders"]["order_id"].nunique())

assert data["orders"]["order_id"].is_unique, (
    "Duplicate order IDs found in the orders dataset."
)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


Total orders: 99441
Unique order IDs: 99441


In [80]:
# Check missing timestamps separately for each order status.
# Missing values are preserved for appropriate handling later.

timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

missing_timestamps_by_status = (
    data["orders"]
    .groupby("order_status", dropna=False)[timestamp_columns]
    .agg(lambda column: column.isna().sum())
)

display(missing_timestamps_by_status)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0,0,2,2,0
canceled,0,141,550,619,0
created,0,5,5,5,0
delivered,0,14,2,8,0
invoiced,0,0,314,314,0
processing,0,0,301,301,0
shipped,0,0,0,1107,0
unavailable,0,0,609,609,0


In [81]:
# Inspect timestamp completeness for delivered orders only.
# These are the orders most relevant to actual delivery-time analysis.

delivered_orders = data["orders"].loc[
    data["orders"]["order_status"].eq("delivered")
]

delivered_missing_dates = (
    delivered_orders[timestamp_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(delivered_missing_dates)

print("Total delivered orders:", len(delivered_orders))

,missing_count
order_purchase_timestamp,0
order_approved_at,14
order_delivered_carrier_date,2
order_delivered_customer_date,8
order_estimated_delivery_date,0


Total delivered orders: 96478


In [82]:
# Identify timestamps that occur out of chronological order.
# These records are flagged for investigation, not deleted or corrected.

orders_check = data["orders"]

carrier_before_purchase = (
    orders_check["order_delivered_carrier_date"].notna()
    & orders_check["order_purchase_timestamp"].notna()
    & (
        orders_check["order_delivered_carrier_date"]
        < orders_check["order_purchase_timestamp"]
    )
)

customer_before_purchase = (
    orders_check["order_delivered_customer_date"].notna()
    & orders_check["order_purchase_timestamp"].notna()
    & (
        orders_check["order_delivered_customer_date"]
        < orders_check["order_purchase_timestamp"]
    )
)

customer_before_carrier = (
    orders_check["order_delivered_customer_date"].notna()
    & orders_check["order_delivered_carrier_date"].notna()
    & (
        orders_check["order_delivered_customer_date"]
        < orders_check["order_delivered_carrier_date"]
    )
)

timestamp_issues = pd.DataFrame({
    "Issue": [
        "Carrier handoff before purchase",
        "Customer delivery before purchase",
        "Customer delivery before carrier handoff"
    ],
    "Affected orders": [
        int(carrier_before_purchase.sum()),
        int(customer_before_purchase.sum()),
        int(customer_before_carrier.sum())
    ]
})

display(timestamp_issues)

,Issue,Affected orders
0,Carrier handoff before purchase,166
1,Customer delivery before purchase,0
2,Customer delivery before carrier handoff,23


In [83]:
# Delivery duration requires a delivered order and both timestamps.
# Missing delivery dates are excluded from this calculation only.

valid_delivery_mask = (
    data["orders"]["order_status"].eq("delivered")
    & data["orders"]["order_purchase_timestamp"].notna()
    & data["orders"]["order_delivered_customer_date"].notna()
)

eligible_orders = data["orders"].loc[valid_delivery_mask]

delivery_days_check = (
    eligible_orders["order_delivered_customer_date"]
    - eligible_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

print(
    "Delivered orders with valid purchase and delivery timestamps:",
    len(delivery_days_check)
)
print("Negative delivery durations:", int((delivery_days_check < 0).sum()))

assert not (delivery_days_check < 0).any(), (
    "Negative purchase-to-delivery durations found."
)

print("Delivery-duration validation passed.")

Delivered orders with valid purchase and delivery timestamps: 96470
Negative delivery durations: 0
Delivery-duration validation passed.


### **Observations**

- The orders dataset contains 99,441 records across eight distinct order statuses.
- A total of 96,478 orders have a `delivered` status; the remaining orders have other statuses, including shipped, canceled, unavailable, invoiced, processing, created, and approved.
- Among delivered orders, 14 approval timestamps, 2 carrier-handoff timestamps, and 8 customer-delivery timestamps are missing.
- A total of 166 orders have carrier-handoff timestamps earlier than their purchase timestamps.
- A total of 23 orders have customer-delivery timestamps earlier than their carrier-handoff timestamps. No customer-delivery timestamps were found earlier than purchase timestamps.
- Of the 96,478 delivered orders, 96,470 have both purchase and customer-delivery timestamps available.
- Validation found zero negative purchase-to-delivery durations among eligible delivered orders.
- All orders are retained. Missing and chronologically inconsistent timestamps are not automatically imputed, corrected, or deleted.
- Delivery metrics will use only the records with valid timestamps required for each calculation. Timestamp inconsistencies will be handled separately for the metrics they affect.
- The original orders dataset remains unchanged, and these findings will guide the creation of delivery-related metrics in subsequent steps.

## **Step 8: Geolocation Data Cleaning**

The geolocation dataset contains multiple latitude and longitude records for the same ZIP code prefix. To make it suitable for analysis, we will aggregate these records by ZIP code prefix and calculate the median latitude and longitude.

The original dataset will remain unchanged, and the aggregated data will be stored separately in `geolocation_cleaned`.

In [84]:
# Aggregate geolocation records by ZIP code prefix using median coordinates.
# Keep the original geolocation dataset unchanged.

geolocation_cleaned = (
    data["geolocation"]
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median")
    )
)

print("Geolocation aggregation completed.")
print("Original rows:", len(data["geolocation"]))
print("Cleaned rows:", len(geolocation_cleaned))

Geolocation aggregation completed.
Original rows: 1000163
Cleaned rows: 19015


In [85]:
# Verify that each ZIP code prefix appears only once.
assert geolocation_cleaned["geolocation_zip_code_prefix"].is_unique, (
    "Duplicate ZIP code prefixes remain."
)

# Check for missing values in the aggregated coordinates.
missing_coordinates = geolocation_cleaned[
    ["geolocation_lat", "geolocation_lng"]
].isna().sum()

print("Missing coordinate values:")
print(missing_coordinates)

assert missing_coordinates.sum() == 0, (
    "Missing latitude or longitude values found."
)

print("All geolocation validation checks passed.")

Missing coordinate values:
geolocation_lat    0
geolocation_lng    0
dtype: int64
All geolocation validation checks passed.


### **Observations**

- The original geolocation dataset contained 1,000,163 records and 19,015 unique ZIP-code prefixes.
- A total of 17,972 ZIP-code prefixes appeared multiple times.
- Latitude and longitude were aggregated using median values for each ZIP-code prefix.
- The resulting `geolocation_cleaned` dataset contains 19,015 records, with one record per ZIP-code prefix.
- Validation confirmed that no duplicate ZIP-code prefixes or missing latitude and longitude values remain in the aggregated dataset.
- The original geolocation dataset remains unchanged, while the aggregated data is stored separately for subsequent analysis.

## **Step 9: Build the Order-Level Dataset**

The orders dataset contains one row per order, while the order-items dataset can contain multiple rows per order. To prevent revenue from being double-counted, we will first aggregate item prices and freight by `order_id`, then merge the results into the orders dataset.

We will also bring in customer identifiers and customer states, along with the cleaned review information. The original datasets will remain unchanged.

In [86]:
# Aggregate item prices and freight to one row per order.
# This prevents duplicate orders and inflated revenue during later joins.

order_values = (
    data["order_items"]
    .groupby("order_id", as_index=False)
    .agg(
        order_value=("price", "sum"),
        order_freight=("freight_value", "sum")
    )
)

# Validate that each order appears only once in the aggregated table.
assert order_values["order_id"].is_unique, (
    "Duplicate order IDs found in order_values."
)

print("Order-level revenue aggregation completed.")
print("Orders with order items:", len(order_values))
display(order_values.head())

Order-level revenue aggregation completed.
Orders with order items: 98666


,order_id,order_value,order_freight
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14


In [87]:
# Select the customer fields needed for order-level analysis.
# customer_unique_id identifies the actual customer across multiple orders.

customer_info = data["customers"][["customer_id", "customer_unique_id", "customer_state"]]

# Each customer_id should map to one customer record.
assert customer_info["customer_id"].is_unique, (
    "Duplicate customer IDs found in customer_info."
)

print("Customer information prepared.")
display(customer_info.head())

Customer information prepared.


,customer_id,customer_unique_id,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,SP


In [88]:
# Start with all original orders, retaining every order status.
orders_cleaned = data["orders"].copy()

# Add customer identifiers and customer state.
orders_cleaned = orders_cleaned.merge(
    customer_info,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Add aggregated item prices and freight.
orders_cleaned = orders_cleaned.merge(
    order_values,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Add the latest cleaned review information.
review_info = reviews_latest[
    ["order_id", "review_score", "is_low_review"]
].copy()

assert review_info["order_id"].is_unique, (
    "Duplicate order IDs found in review_info."
)

orders_cleaned = orders_cleaned.merge(
    review_info,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Calculate total order value including freight.
orders_cleaned["order_value_with_freight"] = (
    orders_cleaned["order_value"] + orders_cleaned["order_freight"]
)

# Flag orders eligible for revenue analysis.
# Keep all orders, but exclude canceled and unavailable orders from revenue metrics.
orders_cleaned["include_in_revenue"] = (
    ~orders_cleaned["order_status"].isin(["canceled", "unavailable"])
)

print("Order-level dataset created.")
print("Original orders:", len(data["orders"]))
print("Cleaned orders:", len(orders_cleaned))

assert len(orders_cleaned) == len(data["orders"]), (
    "Order count changed after merging."
)
assert orders_cleaned["order_id"].is_unique, (
    "Duplicate order IDs found in orders_cleaned."
)
assert orders_cleaned["include_in_revenue"].notna().all(), (
    "Missing revenue eligibility flags found."
)

print("Order-level row-count validation passed.")

print("\nRevenue eligibility by order status:")
display(
    orders_cleaned.groupby("order_status")["include_in_revenue"]
    .value_counts()
    .unstack(fill_value=0)
)

Order-level dataset created.
Original orders: 99441
Cleaned orders: 99441
Order-level row-count validation passed.

Revenue eligibility by order status:


include_in_revenue,False,True
order_status,,
approved,0,2
canceled,625,0
created,0,5
delivered,0,96478
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,609,0


In [89]:
# Compare the cleaned order-level total with the original item-price total.
# Validate after merging to catch any issues introduced by the joins.

raw_item_price_total = data["order_items"]["price"].sum()
cleaned_item_price_total = orders_cleaned["order_value"].sum()

print(f"Raw item-price total:     {raw_item_price_total:.2f}")
print(f"Cleaned order-value total: {cleaned_item_price_total:.2f}")

assert abs(raw_item_price_total - cleaned_item_price_total) < 0.01, (
    "Revenue mismatch: cleaned order values do not match raw order_items."
)

print("Revenue reconciliation passed.")

Raw item-price total:     13591643.70
Cleaned order-value total: 13591643.70
Revenue reconciliation passed.


### **Observations**

- The original orders dataset contains 99,441 records, with one row per order.
- Item prices and freight values were aggregated by `order_id` to create the `order_values` dataset, containing 98,666 orders with order items.
- Customer information, including `customer_unique_id` and `customer_state`, was prepared for integration into the order-level dataset.
- The aggregated order values were merged with the original orders, customer information, and cleaned review information to create `orders_cleaned`.
- The resulting `orders_cleaned` dataset contains 99,441 records, preserving the original order count and maintaining one row per order.
- Revenue validation confirmed that the total item-price value in `orders_cleaned` matches the raw `order_items` total of **R$13,591,643.70**.
- The order IDs in `orders_cleaned` are unique, and the original datasets remain unchanged.
- The `order_value` column represents the total item price excluding freight, while `order_freight` represents total freight charges. `order_value_with_freight` combines both amounts.
- The `include_in_revenue` flag excludes canceled and unavailable orders from revenue analysis while retaining all orders in the cleaned dataset. A total of 1,234 orders are excluded by this rule.
- Orders without matching item or review records are retained, with missing values left unchanged rather than automatically replaced with zero.

## **Step 10: Create Derived Order-Level Columns**

Create additional columns from order timestamps to support monthly trend analysis and delivery-performance analysis. Actual delivery duration will be calculated only for delivered orders with valid purchase and customer-delivery timestamps. Missing or invalid timestamp pairs will remain missing for the metrics they affect.

In [90]:
# Extract the month of purchase for monthly trend analysis.
orders_cleaned["order_month"] = (
    orders_cleaned["order_purchase_timestamp"].dt.to_period("M")
)

# Actual delivery duration: purchase to customer delivery.
# Only delivered orders with both timestamps are eligible.
valid_delivery = (
    orders_cleaned["order_status"].eq("delivered")
    & orders_cleaned["order_purchase_timestamp"].notna()
    & orders_cleaned["order_delivered_customer_date"].notna()
)

orders_cleaned["delivery_days"] = (
    orders_cleaned["order_delivered_customer_date"]
    - orders_cleaned["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders_cleaned.loc[~valid_delivery, "delivery_days"] = float("nan")

# Estimated delivery duration: purchase to estimated delivery date.
# Missing timestamps naturally produce missing duration values.
orders_cleaned["estimated_days"] = (
    orders_cleaned["order_estimated_delivery_date"]
    - orders_cleaned["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Delivery delay: actual customer delivery minus estimated delivery.
# Positive means late; zero or negative means on time or early.
valid_delay = (
    valid_delivery
    & orders_cleaned["order_estimated_delivery_date"].notna()
)

orders_cleaned["delay_days"] = (
    orders_cleaned["order_delivered_customer_date"]
    - orders_cleaned["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

orders_cleaned.loc[~valid_delay, "delay_days"] = float("nan")

print("Purchase month and delivery metrics created.")

Purchase month and delivery metrics created.


In [91]:
# Start with missing values because some orders have no valid delay.
orders_cleaned["is_late"] = float("nan")

valid_delay = orders_cleaned["delay_days"].notna()

# Mark an order as late (1) only when actual delivery was after the estimate.
# Orders delivered on time or early receive 0.
orders_cleaned.loc[valid_delay, "is_late"] = (
    orders_cleaned.loc[valid_delay, "delay_days"] > 0
).astype(int)

# Group valid delivery delays into useful business categories.
orders_cleaned["delay_bucket"] = pd.NA

orders_cleaned.loc[
    valid_delay & orders_cleaned["delay_days"].le(0),
    "delay_bucket"
] = "on time / early"

orders_cleaned.loc[
    valid_delay
    & orders_cleaned["delay_days"].gt(0)
    & orders_cleaned["delay_days"].le(3),
    "delay_bucket"
] = "1-3 days late"

orders_cleaned.loc[
    valid_delay
    & orders_cleaned["delay_days"].gt(3)
    & orders_cleaned["delay_days"].le(7),
    "delay_bucket"
] = "4-7 days late"

orders_cleaned.loc[
    valid_delay & orders_cleaned["delay_days"].gt(7),
    "delay_bucket"
] = "8+ days late"

print("Late-delivery flags and delay categories created.")

Late-delivery flags and delay categories created.


In [92]:
# Seller handling time: order approval to carrier handoff.
# Exclude missing timestamps and handoffs that occurred before approval.
valid_handling = (
    orders_cleaned["order_approved_at"].notna()
    & orders_cleaned["order_delivered_carrier_date"].notna()
    & (
        orders_cleaned["order_delivered_carrier_date"]
        >= orders_cleaned["order_approved_at"]
    )
)

orders_cleaned["seller_handling_days"] = (
    orders_cleaned["order_delivered_carrier_date"]
    - orders_cleaned["order_approved_at"]
).dt.total_seconds() / 86400

orders_cleaned.loc[~valid_handling, "seller_handling_days"] = float("nan")

# Carrier transit time: carrier handoff to customer delivery.
# Use delivered orders with valid timestamps in chronological order.
valid_transit = (
    orders_cleaned["order_status"].eq("delivered")
    & orders_cleaned["order_delivered_carrier_date"].notna()
    & orders_cleaned["order_delivered_customer_date"].notna()
    & (
        orders_cleaned["order_delivered_customer_date"]
        >= orders_cleaned["order_delivered_carrier_date"]
    )
)

orders_cleaned["carrier_transit_days"] = (
    orders_cleaned["order_delivered_customer_date"]
    - orders_cleaned["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

orders_cleaned.loc[~valid_transit, "carrier_transit_days"] = float("nan")

print("Seller handling and carrier transit metrics created.")

Seller handling and carrier transit metrics created.


In [93]:
# Calculate freight as a proportion of item price.
# Zero item prices produce a missing ratio rather than division by zero.
orders_cleaned["freight_ratio"] = (
    orders_cleaned["order_freight"]
    / orders_cleaned["order_value"].replace(0, float("nan"))
)

# Flag purchases from January 2017 through August 2018.
# Keep all orders; this flag identifies the period used for trend analysis.
orders_cleaned["in_analysis_period"] = (
    orders_cleaned["order_purchase_timestamp"].ge("2017-01-01")
    & orders_cleaned["order_purchase_timestamp"].lt("2018-09-01")
)

print("Freight ratio and analysis-period flag created.")
print("Orders inside analysis period:", orders_cleaned["in_analysis_period"].sum())
print("Orders outside analysis period:", (~orders_cleaned["in_analysis_period"]).sum())

Freight ratio and analysis-period flag created.
Orders inside analysis period: 99092
Orders outside analysis period: 349


In [94]:
# Check that all required order-level columns exist.
required_columns = [
    "order_month",
    "delivery_days",
    "estimated_days",
    "delay_days",
    "is_late",
    "delay_bucket",
    "seller_handling_days",
    "carrier_transit_days",
    "order_value",
    "order_freight",
    "order_value_with_freight",
    "freight_ratio",
    "is_low_review",
    "customer_state",
    "customer_unique_id",
    "in_analysis_period",
    "include_in_revenue"
]

missing_columns = [
    column for column in required_columns
    if column not in orders_cleaned.columns
]

assert not missing_columns, f"Missing required columns: {missing_columns}"

# Confirm that the original order count and unique order IDs are preserved.
assert len(orders_cleaned) == len(data["orders"])
assert orders_cleaned["order_id"].is_unique

# Confirm that calculated durations contain no negative values.
# Invalid timestamp pairs should already be missing from the relevant metrics.
for column in [
    "delivery_days",
    "estimated_days",
    "seller_handling_days",
    "carrier_transit_days"
]:
    assert not (orders_cleaned[column].dropna() < 0).any(), (
        f"Negative values found in {column}."
    )

# Late-delivery flags must be either 0, 1, or missing.
assert orders_cleaned["is_late"].dropna().isin([0, 1]).all()

print("All derived-column validation checks passed.")
print("Total orders:", len(orders_cleaned))
print("Valid delivery durations:", orders_cleaned["delivery_days"].notna().sum())

All derived-column validation checks passed.
Total orders: 99441
Valid delivery durations: 96470


### **Observations**

- Created delivery-related columns to analyze monthly trends, actual delivery duration, estimated delivery duration, and delivery delays.
- Added `is_late` and `delay_bucket` to categorize delivery performance.
- Calculated `seller_handling_days` and `carrier_transit_days`, excluding invalid timestamp pairs from the relevant metrics.
- Created `freight_ratio` and `in_analysis_period` for shipping-cost and trend analysis.
- Preserved customer information, review indicators, and all original orders.
- Validation passed: `orders_cleaned` contains 99,441 unique orders, including 96,470 valid delivery durations.
- The analysis period contains 99,092 orders, while 349 orders fall outside it. No orders were removed.

## **Step 11: Build the Item-Level Dataset**

Create `order_items_cleaned` by combining the original order items with seller and customer state information. Add the `same_state` flag to identify whether each seller and customer are located in the same state. Preserve all original item records and validate the row count, state information, and item-price total to ensure the joins do not introduce data duplication or revenue discrepancies.

In [95]:
# Start with the original order items, preserving every item row.
order_items_cleaned = data["order_items"].copy()

# Add the seller's state to each item using seller_id.
seller_info = data["sellers"][["seller_id", "seller_state"]]

order_items_cleaned = order_items_cleaned.merge(
    seller_info,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

# Get each order's customer state through the orders and customers tables.
customer_state_info = (
    data["orders"][["order_id", "customer_id"]]
    .merge(
        data["customers"][["customer_id", "customer_state"]],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
    [["order_id", "customer_state"]]
)

# Add the customer's state to each item.
order_items_cleaned = order_items_cleaned.merge(
    customer_state_info,
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Flag whether the seller and customer are in the same state.
order_items_cleaned["same_state"] = (
    order_items_cleaned["seller_state"]
    == order_items_cleaned["customer_state"]
).astype("int8")

print("Item-level dataset created.")
print("Original item rows:", len(data["order_items"]))
print("Cleaned item rows:", len(order_items_cleaned))

Item-level dataset created.
Original item rows: 112650
Cleaned item rows: 112650


In [96]:
# Confirm that every original item row has been preserved.
assert len(order_items_cleaned) == 112650, (
    "Unexpected item row count."
)

# Confirm that seller and customer states are available for every item.
assert order_items_cleaned["seller_state"].notna().all(), (
    "Missing seller states found."
)

assert order_items_cleaned["customer_state"].notna().all(), (
    "Missing customer states found."
)

# Confirm that the merge did not change the total item-price value.
raw_price_total = data["order_items"]["price"].sum()
cleaned_price_total = order_items_cleaned["price"].sum()

assert abs(raw_price_total - cleaned_price_total) < 0.01, (
    "Item-price total changed after merging."
)

# Confirm that same_state contains only 0 or 1.
assert order_items_cleaned["same_state"].isin([0, 1]).all()

print("Item-level validation passed.")
print("Item rows:", len(order_items_cleaned))
print(f"Raw item-price total: {raw_price_total:.2f}")
print(f"Cleaned item-price total: {cleaned_price_total:.2f}")

display(
    order_items_cleaned[
        ["order_id", "seller_id", "seller_state", "customer_state", "same_state"]
    ].head()
)

Item-level validation passed.
Item rows: 112650
Raw item-price total: 13591643.70
Cleaned item-price total: 13591643.70


,order_id,seller_id,seller_state,customer_state,same_state
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,SP,RJ,0
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,SP,SP,1
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,MG,MG,1
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,SP,SP,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,PR,SP,0


### **Observations**

- Created `order_items_cleaned` by merging the original order items with seller and customer state information.
- Added `seller_state`, `customer_state`, and `same_state` for geographic and seller-performance analysis.
- Preserved all 112,650 item records without increasing or reducing the row count.
- Validation confirmed that the item-price total matches the original `order_items` total of R$13,591,643.70.
- The cleaned item-level dataset is ready for subsequent analysis.

## **Step 12: Save the Cleaned Datasets**

In this step, the cleaned datasets are exported as CSV files to the `data/processed/` directory. The original raw files remain unchanged, allowing the cleaning process to be reproduced whenever needed.

The eight datasets saved are:

- `orders_cleaned.csv`
- `order_items_cleaned.csv`
- `payments.csv`
- `reviews_latest.csv`
- `products.csv`
- `sellers.csv`
- `customers.csv`
- `geolocation_cleaned.csv`

All files are saved with `index=False` to avoid adding an unnecessary index column. These processed datasets will be used in the subsequent stages of the Olist E-Commerce Analytics project.

In [97]:
# Convert flags and review scores to nullable integers.
# Missing values remain missing.

orders_cleaned["is_late"] = orders_cleaned["is_late"].astype("Int8")
orders_cleaned["is_low_review"] = orders_cleaned["is_low_review"].astype("Int8")
orders_cleaned["review_score"] = orders_cleaned["review_score"].astype("Int8")

reviews_latest["is_low_review"] = reviews_latest["is_low_review"].astype("Int8")
reviews_latest["review_score"] = reviews_latest["review_score"].astype("Int8")

print("Nullable integer columns updated successfully.")
print(orders_cleaned[["is_late", "is_low_review", "review_score"]].dtypes)

Nullable integer columns updated successfully.
is_late          Int8
is_low_review    Int8
review_score     Int8
dtype: object


In [98]:
# Save the cleaned datasets as CSV files without DataFrame indexes.
cleaned_datasets = {
    "orders_cleaned": orders_cleaned,
    "order_items_cleaned": order_items_cleaned,
    "payments": data["payments"],
    "reviews_latest": reviews_latest,
    "products": data["products"],
    "sellers": data["sellers"],
    "customers": data["customers"],
    "geolocation_cleaned": geolocation_cleaned
}

for name, df in cleaned_datasets.items():
    output_path = os.path.join(processed_path, f"{name}.csv")
    df.to_csv(output_path, index=False)
    print(f"Saved {name}.csv - {len(df)} rows")

print("\nAll cleaned datasets saved successfully.")

Saved orders_cleaned.csv - 99441 rows
Saved order_items_cleaned.csv - 112650 rows
Saved payments.csv - 103886 rows
Saved reviews_latest.csv - 98673 rows
Saved products.csv - 32951 rows
Saved sellers.csv - 3095 rows
Saved customers.csv - 99441 rows
Saved geolocation_cleaned.csv - 19015 rows

All cleaned datasets saved successfully.


In [99]:
# Confirm that every expected CSV file exists in data/processed/.
for name, df in cleaned_datasets.items():
    file_path = os.path.join(processed_path, f"{name}.csv")

    assert os.path.isfile(file_path), f"File not found: {file_path}"

print("All eight processed CSV files exist.")

All eight processed CSV files exist.


### **Observation**

- Successfully exported all 8 processed datasets to the `data/processed/` directory.
- Saved `orders_cleaned` (99,441 rows), `order_items_cleaned` (112,650 rows), `payments` (103,886 rows), `reviews_latest` (98,673 rows), `products` (32,951 rows), `sellers` (3,095 rows), `customers` (99,441 rows), and `geolocation_cleaned` (19,015 rows).
- All files were saved with `index=False` to avoid unnecessary index columns.
- Nullable integer columns (`is_late`, `is_low_review`, and `review_score`) were prepared using Pandas' `Int8` data type to preserve whole-number values while allowing missing values.
- The processed datasets are ready for the next stage of the Olist E-Commerce Analytics project.

## **Step 13: Validate the Saved Datasets**

In this step, the exported CSV files are checked to ensure they exist and contain the expected number of rows. The saved datasets are also checked for key data integrity requirements, including unique order IDs and reconciliation of the total item price with the original dataset. These checks help confirm that the cleaned datasets are ready for the next stage of the Olist E-Commerce Analytics project.

In [100]:
# Validate the saved CSV files and their expected row counts.

expected_rows = {
    "orders_cleaned": 99_441,
    "order_items_cleaned": 112_650,
    "reviews_latest": 98_673,
    "geolocation_cleaned": 19_015,
}

for name, df in cleaned_datasets.items():
    file_path = os.path.join(processed_path, f"{name}.csv")

    assert os.path.isfile(file_path), f"{name}.csv was not found."

    saved_df = pd.read_csv(file_path)

    if name in expected_rows:
        assert len(saved_df) == expected_rows[name], (
            f"{name}: expected {expected_rows[name]} rows, "
            f"but found {len(saved_df)}."
        )

    print(f"{name}.csv: {len(saved_df):,} rows - verified")

# Check order-level uniqueness.
assert orders_cleaned["order_id"].is_unique

# Check item-price reconciliation using the saved orders dataset.
saved_orders = pd.read_csv(
    os.path.join(processed_path, "orders_cleaned.csv")
)

raw_item_price_total = data["order_items"]["price"].sum()
saved_item_price_total = saved_orders["order_value"].sum()

assert abs(raw_item_price_total - saved_item_price_total) < 0.01

print(f"\nRaw item-price total: R${raw_item_price_total:,.2f}")
print(f"Saved item-price total: R${saved_item_price_total:,.2f}")
print("\nAll validation checks passed!")

orders_cleaned.csv: 99,441 rows - verified
order_items_cleaned.csv: 112,650 rows - verified
payments.csv: 103,886 rows - verified
reviews_latest.csv: 98,673 rows - verified
products.csv: 32,951 rows - verified
sellers.csv: 3,095 rows - verified
customers.csv: 99,441 rows - verified
geolocation_cleaned.csv: 19,015 rows - verified

Raw item-price total: R$13,591,643.70
Saved item-price total: R$13,591,643.70

All validation checks passed!


### **Observation**

- Verified that all 8 processed CSV files exist and contain the expected number of rows.
- Confirmed that the datasets contain 99,441 orders, 112,650 order items, 98,673 latest reviews, and 19,015 aggregated geolocation records, along with the remaining supporting datasets.
- Verified that `orders_cleaned` contains unique order IDs.
- Compared the total item price against the original `order_items` dataset. Both totals matched at **R$13,591,643.70**.
- All validation checks passed, confirming that the processed files were saved correctly and the total item-price value was preserved.

# **Table Relationships**

The Olist processed datasets are connected through several shared identifiers.


| File | Grain | Main Key | Use / What Changed |
|---|---|---|---|
| `orders_cleaned` | One order | `order_id` | Order-level analysis: delivery, revenue, reviews, customer state. Contains all derived columns and flags |
| `order_items_cleaned` | One item within an order | `order_id` + `order_item_id` | Seller, product, and state analysis. Adds `seller_state`, `customer_state`, `same_state` |
| `reviews_latest` | One review per order | `order_id` | Latest review only. Adds `is_low_review` and the early-review flag; comment text removed |
| `products` | One product | `product_id` | Adds `product_category_english`; zero weights set to NaN |
| `payments` | One payment record | `order_id` + `payment_sequential` | Unchanged |
| `customers` | One record per order | `customer_id` | Unchanged; `customer_unique_id` identifies repeat customers |
| `sellers` | One seller | `seller_id` | Unchanged |
| `geolocation_cleaned` | One ZIP prefix | `geolocation_zip_code_prefix` | Median latitude and longitude |

### **Important Join Considerations**

- Use `orders_cleaned` for order-level metrics and `order_items_cleaned` for seller, product, and state analysis. Do not join them directly, or item rows will duplicate order values.
- 1,278 orders have multiple sellers, so seller-level analysis must use `order_items_cleaned`.
- 2,961 orders have multiple payment records. Aggregate `payments` by `order_id` before joining it to orders.
- Use `include_in_revenue` for revenue, `in_analysis_period` for trends, and non-missing `delivery_days` for delivery analysis.
- `is_late`, `is_low_review`, and `review_score` are missing (not 0) for orders without valid delivery data or a review. Do not treat NaN as "on time" or "good review".
- Use `customer_unique_id`, not `customer_id`, for repeat-customer analysis.
- `geolocation_cleaned` joins to customers and sellers on ZIP prefix. It is saved but not yet used, because the distance column is optional.


# **Final Summary**

I cleaned and prepared the nine Olist datasets and saved 8 processed files. No orders were removed.

Key results:

- Timestamps were converted. 96,470 of 96,478 delivered orders have a valid delivery duration; the other 8 lack a customer-delivery date and stay missing. 14 delivered orders lack an approval date and 2 lack a carrier date; these are excluded only from the metrics they affect.
- 166 orders have carrier handoff before purchase and 23 have delivery before handoff. They are excluded only from handling and transit metrics.
- 610 products have an unknown category; 2 categories were translated manually. 4 zero weights were set to NaN.
- 551 duplicate reviews were removed (98,673 remain). 71 early reviews remain flagged (74 before deduplication).
- 768 orders have no review and stay missing.
- Geolocation was reduced to 19,015 ZIP prefixes.
- 1,234 canceled or unavailable orders are excluded from revenue by `include_in_revenue`. 349 orders fall outside Jan 2017 to Aug 2018.
- Item-price totals match the raw data at R$13,591,643.70 in `orders_cleaned`, `order_items_cleaned`, and the saved CSV.

## **Next Step**
The next stage will load the processed files into PostgreSQL:

- Create tables with primary and foreign keys
- Load the CSVs from `data/processed/`
- Check row counts against this notebook
- Build views for the SQL business analysis

> First understand the data. Then clean it. Then analyze it.